<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module2_Labs/Lab6_Full_QAOA_MaxCut.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 6 — Full QAOA on the 5-Node Max-Cut Problem
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---
## Learning Objectives
1. Combine all QAOA components: $H^{\otimes 5}$ → $U_C(\gamma)$ → $U_B(\beta)$ → repeat
2. Add the **Mixer operator** $U_B(\beta) = \prod_i R_X^{(i)}(\beta)$
3. Run a **1-layer QAOA** and observe improvement over random guessing
4. Run **multi-layer QAOA** (p=1,2,3,5) and compare quality
5. *(Optional)* Run on **real IBM Quantum hardware** and compare to simulator

---
### 📖 The Full QAOA Circuit

$$|\varphi(\gamma,\beta)\rangle = U_B(\beta_p)U_C(\gamma_p)\cdots U_B(\beta_1)U_C(\gamma_1)H^{\otimes n}|0\rangle^n$$

Where:
- $U_C(\gamma) = \prod_{(i,j)\in E} R_{ZZ}(-\gamma)$ — encodes edge cut values into phase
- $U_B(\beta) = \prod_i R_X^{(i)}(\beta)$ — converts phase differences to amplitude differences
- $p$ layers repeated with potentially different $\gamma_k, \beta_k$ per layer

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import networkx as nx
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

simulator = AerSimulator()

N_NODES = 5
EDGES   = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

def cut_value(bitstring, edges):
    return sum(1 for u,v in edges if bitstring[u] != bitstring[v])

all_bitstrings = [format(i, '05b') for i in range(32)]

print("Setup complete.")
print(f"Graph: {N_NODES} nodes, {len(EDGES)} edges")
print(f"Optimal cut = 6, achieved by: 00011 and 11100")

---
## Part 1: Build the Complete QAOA Circuit

In [ ]:
# ── 1.1  QAOA circuit builder ─────────────────────────────────────────────────
def build_qaoa_circuit(gammas, betas, n_qubits, edges, measure=True):
    """
    Build the full QAOA circuit for p layers.
    gammas, betas: lists of length p (one value per layer)
    """
    p = len(gammas)
    assert len(betas) == p, "Must have same number of gammas and betas"

    qc = QuantumCircuit(n_qubits, n_qubits if measure else 0)

    # Step 1: Initial superposition — "Everyone on Stage"
    qc.h(range(n_qubits))

    for layer in range(p):
        # Step 2: Cost Operator U_C(γ) — "Judges' Scores"
        for u, v in edges:
            qc.cx(u, v)
            qc.rz(-gammas[layer], v)
            qc.cx(u, v)

        # Step 3: Mixer Operator U_B(β) — "Crowd's Applause"
        for i in range(n_qubits):
            qc.rx(2 * betas[layer], i)

    if measure:
        qc.measure(range(n_qubits), range(n_qubits))

    return qc

# Draw a 1-layer QAOA circuit
qc_demo = build_qaoa_circuit([np.pi/3], [np.pi/4], N_NODES, EDGES, measure=False)
print(f"1-layer QAOA circuit ({qc_demo.num_qubits} qubits, {qc_demo.depth()} depth, {qc_demo.count_ops()} ops):")
print(qc_demo.draw('text'))

In [ ]:
# ── 1.2  Compute expected cut value F(γ,β) ────────────────────────────────────
def compute_expectation(gammas, betas, n_qubits, edges, shots=2000):
    """
    Run QAOA circuit and compute ⟨H_C⟩ = expected cut value.
    Returns a float (higher is better for Max-Cut).
    """
    qc = build_qaoa_circuit(gammas, betas, n_qubits, edges, measure=True)
    compiled = transpile(qc, simulator)
    result   = simulator.run(compiled, shots=shots).result()
    counts   = result.get_counts()

    expected = 0.0
    total    = sum(counts.values())
    for bs, cnt in counts.items():
        cv       = cut_value(bs, edges)
        expected += cv * (cnt / total)
    return expected

# Test with fixed γ=π/3, β=π/4
F_test = compute_expectation([np.pi/3], [np.pi/4], N_NODES, EDGES)
random_expected = sum(cut_value(bs,EDGES) for bs in all_bitstrings) / 32

print(f"Expected cut value:")
print(f"  Random (no QAOA):          {random_expected:.3f}")
print(f"  QAOA(p=1, γ=π/3, β=π/4):  {F_test:.3f}")
print(f"  Maximum possible:          6.000")

---
## Part 2: Scanning the γ-β Landscape (p=1)

In [ ]:
# ── 2.1  2D scan of F(γ,β) for p=1 ─────────────────────────────────────────
print("Scanning γ-β landscape (this may take ~30 seconds)...")

# Use statevector (exact, fast) for the scan
def compute_expectation_exact(gamma, beta, n_qubits, edges):
    qc = build_qaoa_circuit([gamma], [beta], n_qubits, edges, measure=False)
    sv = Statevector(qc)
    exp_val = 0.0
    for bs, amp in zip(all_bitstrings, sv.data):
        exp_val += cut_value(bs, edges) * abs(amp)**2
    return exp_val

n_pts = 30
gammas_scan = np.linspace(0.01, np.pi, n_pts)
betas_scan  = np.linspace(0.01, np.pi, n_pts)

F_grid = np.zeros((n_pts, n_pts))
for i, g in enumerate(gammas_scan):
    for j, b in enumerate(betas_scan):
        F_grid[i, j] = compute_expectation_exact(g, b, N_NODES, EDGES)

# Find best
best_idx = np.unravel_index(F_grid.argmax(), F_grid.shape)
best_gamma = gammas_scan[best_idx[0]]
best_beta  = betas_scan[best_idx[1]]
best_F     = F_grid[best_idx]

print(f"Best F(γ,β) from grid search: {best_F:.4f}")
print(f"  γ* = {best_gamma:.4f} rad ({np.degrees(best_gamma):.1f}°)")
print(f"  β* = {best_beta:.4f} rad ({np.degrees(best_beta):.1f}°)")

In [ ]:
# ── 2.2  Plot the landscape ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.contourf(np.degrees(betas_scan), np.degrees(gammas_scan), F_grid,
                 levels=20, cmap='viridis')
plt.colorbar(im, ax=ax, label='Expected cut value ⟨H_C⟩')
ax.scatter([np.degrees(best_beta)], [np.degrees(best_gamma)],
           color='red', s=100, zorder=5, label=f'Best: F={best_F:.3f}')
ax.set_xlabel('β (degrees)'); ax.set_ylabel('γ (degrees)')
ax.set_title('QAOA p=1 Landscape: ⟨H_C⟩(γ,β)\nBrighter = higher expected cut value')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 3: Classical Optimization with COBYLA

In [ ]:
# ── 3.1  COBYLA Optimizer ────────────────────────────────────────────────────
def run_qaoa_optimization(p_layers=1, n_shots=2000, verbose=True):
    """
    Run QAOA with COBYLA optimization for p layers.
    Returns: (best_params, best_F, history)
    """
    n_params = 2 * p_layers  # p gammas + p betas
    history  = []
    eval_count = [0]

    def objective(params):
        gammas = params[:p_layers]
        betas  = params[p_layers:]
        F = compute_expectation(gammas, betas, N_NODES, EDGES, shots=n_shots)
        history.append(F)
        eval_count[0] += 1
        return -F  # negative because COBYLA minimizes

    # Random initial parameters
    np.random.seed(42)
    init_params = np.random.uniform(0, np.pi, n_params)

    if verbose:
        print(f"Starting COBYLA optimization (p={p_layers}, n_shots={n_shots})...")
        print(f"  Initial params: γ={np.round(init_params[:p_layers],3)}, β={np.round(init_params[p_layers:],3)}")
        print(f"  Initial F = {-objective(init_params):.4f}")

    result = minimize(
        objective, init_params,
        method='COBYLA',
        options={'maxiter': 300, 'rhobeg': 0.5}
    )

    best_params = result.x
    best_F      = -result.fun

    if verbose:
        print(f"  Optimized in {eval_count[0]} evaluations")
        print(f"  Best F = {best_F:.4f}")
        gammas_opt = best_params[:p_layers]
        betas_opt  = best_params[p_layers:]
        print(f"  γ* = {np.round(gammas_opt,3)} rad")
        print(f"  β* = {np.round(betas_opt,3)} rad")

    return best_params, best_F, history

# Run p=1 optimization
params_p1, F_p1, history_p1 = run_qaoa_optimization(p_layers=1, n_shots=1000)

In [ ]:
# ── 3.2  Plot optimization convergence ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_p1, 'b-o', markersize=3, alpha=0.7)
ax.axhline(6, color='green', linestyle='--', lw=2, label='Optimal cut = 6')
ax.axhline(sum(cut_value(bs,EDGES) for bs in all_bitstrings)/32,
           color='gray', linestyle='--', lw=1.5, label=f'Random = {sum(cut_value(bs,EDGES) for bs in all_bitstrings)/32:.2f}')
ax.set_xlabel('COBYLA evaluation'); ax.set_ylabel('Expected cut value F')
ax.set_title('COBYLA Convergence for QAOA p=1')
ax.set_ylim(0, 7); ax.legend()
plt.tight_layout(); plt.show()
print(f"\nFinal F(p=1) = {F_p1:.4f}  (vs. random = 3.00, optimal = 6.00)")

In [ ]:
# ── 3.3  Final measurement with optimized parameters ──────────────────────────
gammas_opt = params_p1[:1]
betas_opt  = params_p1[1:]

qc_final = build_qaoa_circuit(gammas_opt, betas_opt, N_NODES, EDGES, measure=True)
compiled  = transpile(qc_final, simulator)
result    = simulator.run(compiled, shots=2000).result()
counts    = result.get_counts()

# Sort by count
top_results = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 measurement outcomes:")
print(f"{'Bitstring':>12} | {'Count':>6} | {'Prob':>6} | {'Cut value':>10}")
print("-" * 45)
for bs, cnt in top_results:
    prob = cnt / 2000
    cv   = cut_value(bs, EDGES)
    opt  = '← OPTIMAL' if cv == 6 else ''
    print(f"{bs:>12} | {cnt:>6} | {prob:>6.4f} | {cv:>10}  {opt}")

# Probability of finding optimal solution
p_optimal = sum(cnt for bs, cnt in counts.items() if cut_value(bs,EDGES)==6) / 2000
print(f"\nP(optimal cut=6) = {p_optimal:.4f}")
print(f"P(optimal) random = {2/32:.4f} (only 2 of 32 bitstrings are optimal)")
print(f"QAOA improvement factor: {p_optimal / (2/32):.1f}x")

In [ ]:
# ── 3.4  Plot measurement distribution ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

sorted_bs  = sorted(counts.keys())
sorted_cnt = [counts.get(bs,0) for bs in sorted_bs]
bar_colors = plt.cm.RdYlGn([cut_value(bs,EDGES)/6 for bs in sorted_bs])

bars = ax.bar(range(len(sorted_bs)), sorted_cnt, color=bar_colors)
ax.axhline(2000/32, color='blue', linestyle='--', lw=1.5, label='Random expectation')

# Mark optimal bitstrings
for i, bs in enumerate(sorted_bs):
    if cut_value(bs,EDGES) == 6:
        ax.text(i, sorted_cnt[i]+5, '★', ha='center', fontsize=10, color='gold')

ax.set_xticks(range(len(sorted_bs)))
ax.set_xticklabels(sorted_bs, rotation=90, fontsize=7)
ax.set_xlabel('Bitstring'); ax.set_ylabel('Count (out of 2000 shots)')
ax.set_title('QAOA Measurement Distribution (Green=high cut, Red=low cut, ★=optimal)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 4: Multi-Layer QAOA (p=1,2,3)

In [ ]:
# ── 4.1  Compare p=1,2,3 layers ──────────────────────────────────────────────
print("Comparing QAOA performance across p layers...")
print("(Each takes ~30-60 seconds — please wait)\n")

results_by_p = {}
for p in [1, 2, 3]:
    print(f"--- p={p} ---")
    params, F, history = run_qaoa_optimization(p_layers=p, n_shots=800, verbose=True)
    results_by_p[p] = {'params': params, 'F': F, 'history': history}
    print()

In [ ]:
# ── 4.2  Compare p=1,2,3 convergence ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence curves
colors_p = {1:'blue', 2:'green', 3:'red'}
for p, data in results_by_p.items():
    axes[0].plot(data['history'], color=colors_p[p],
                 alpha=0.8, label=f'p={p}, F*={data["F"]:.3f}')
axes[0].axhline(6, color='black', linestyle='--', lw=1.5, label='Optimal=6')
axes[0].axhline(3, color='gray', linestyle=':', lw=1, label='Random=3')
axes[0].set_xlabel('Function evaluations')
axes[0].set_ylabel('Expected cut value')
axes[0].set_title('Optimization Convergence by Number of Layers')
axes[0].legend()
axes[0].set_ylim(0, 7)

# Bar chart: final F values
p_vals = list(results_by_p.keys())
F_vals = [results_by_p[p]['F'] for p in p_vals]
axes[1].bar([f'p={p}' for p in p_vals], F_vals,
             color=[colors_p[p] for p in p_vals], alpha=0.8)
axes[1].axhline(6, color='green', linestyle='--', lw=2, label='Optimal=6')
axes[1].axhline(3, color='gray', linestyle=':', lw=1.5, label='Random=3')
axes[1].set_ylabel('Best expected cut value')
axes[1].set_title('Final Performance vs. Number of Layers')
axes[1].set_ylim(0, 7)
axes[1].legend()
for i, (p, F) in enumerate(zip(p_vals, F_vals)):
    axes[1].text(i, F+0.1, f'{F:.3f}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

---
### ✏️ Exercise 6.1 — Probability of Success vs. Layers

For each p=1,2,3:
1. Run the optimized circuit for 3000 shots
2. Compute P(optimal) = probability of measuring cut=6
3. Compute P(good) = probability of measuring cut ≥ 5
4. Plot how these probabilities improve with p

In [ ]:
# YOUR CODE HERE
p_optimal_by_p  = []
p_good_by_p     = []
SHOTS = 3000

for p, data in results_by_p.items():
    params = data['params']
    gammas = params[:p]
    betas  = params[p:]
    qc = build_qaoa_circuit(gammas, betas, N_NODES, EDGES, measure=True)
    result = simulator.run(transpile(qc, simulator), shots=SHOTS).result()
    counts = result.get_counts()

    p_opt  = sum(cnt for bs,cnt in counts.items() if cut_value(bs,EDGES)==6) / SHOTS
    p_good = sum(cnt for bs,cnt in counts.items() if cut_value(bs,EDGES)>=5) / SHOTS
    p_optimal_by_p.append(p_opt)
    p_good_by_p.append(p_good)
    print(f"p={p}: P(cut=6)={p_opt:.4f}, P(cut≥5)={p_good:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
x = list(results_by_p.keys())
ax.plot(x, p_optimal_by_p, 'go-', ms=8, lw=2, label='P(cut=6) = optimal')
ax.plot(x, p_good_by_p,    'bs-', ms=8, lw=2, label='P(cut≥5) = good')
ax.axhline(2/32, color='gray', linestyle='--', label='P(optimal) random = 0.0625')
ax.set_xlabel('Number of QAOA layers (p)')
ax.set_ylabel('Probability')
ax.set_title('Probability of Finding Optimal Solution vs. QAOA Depth')
ax.legend(); ax.set_xticks(x)
plt.tight_layout()
plt.show()

---
## ⭐ Optional: Run on Real IBM Quantum Hardware

In [ ]:
USE_REAL_HARDWARE = False  # Set True if you have IBM Quantum access

if USE_REAL_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
    print(f"Hardware backend: {backend.name}")
    print(f"  Qubits: {backend.configuration().n_qubits}")
    print(f"  Basis gates: {backend.configuration().basis_gates}")
else:
    print("Simulator mode. Set USE_REAL_HARDWARE=True for IBM Quantum.")

In [ ]:
if USE_REAL_HARDWARE:
    # Use optimized p=1 parameters
    gammas_hw = params_p1[:1]
    betas_hw  = params_p1[1:]
    qc_hw     = build_qaoa_circuit(gammas_hw, betas_hw, N_NODES, EDGES, measure=True)

    # Transpile to hardware native gates
    qc_hw_compiled = transpile(qc_hw, backend, optimization_level=3)
    print(f"Circuit depth after transpilation: {qc_hw_compiled.depth()}")
    print(f"Gate count: {qc_hw_compiled.count_ops()}")

    sampler = Sampler(backend)
    hw_job  = sampler.run([qc_hw_compiled], shots=1000)
    print(f"Job ID: {hw_job.job_id()}")
    print("Waiting for hardware results (may take a few minutes)...")

    hw_result = hw_job.result()
    hw_counts = hw_result[0].data.c.get_counts()

    # Compare simulator vs hardware
    sim_result = simulator.run(
        transpile(build_qaoa_circuit(gammas_hw, betas_hw, N_NODES, EDGES, measure=True), simulator),
        shots=1000
    ).result()
    sim_counts = sim_result.get_counts()

    print("\nTop outcomes comparison:")
    print(f"{'Bitstring':>12} | {'Simulator':>10} | {'Hardware':>10} | {'Cut':>5}")
    all_bs = sorted(set(list(sim_counts.keys()) + list(hw_counts.keys())))
    for bs in sorted(all_bs, key=lambda x: sim_counts.get(x,0), reverse=True)[:10]:
        cv = cut_value(bs, EDGES)
        print(f"{bs:>12} | {sim_counts.get(bs,0):>10} | {hw_counts.get(bs,0):>10} | {cv:>5}")

---
## ✅ Lab 6 Summary

| Component | Circuit | Role |
|-----------|---------|------|
| Step 1 | $H^{\otimes 5}$ | Equal superposition of all 32 candidates |
| Step 2 (U_C) | 6×(CNOT–R_Z–CNOT) | Encode cut value into phase |
| Step 3 (U_B) | 5×R_X(β) | Convert phase → amplitude bias |
| Repeat | p layers | Sharpen the distribution |
| Step 5 | COBYLA | Find optimal γ and β |

**Results observed:**
- p=1 achieves F ≈ 4-5 (vs random F=3)
- p=2,3 approach F ≈ 5-6
- Optimal bitstrings (00011, 11100) get significantly boosted probability

## 🔭 Preview of Lab 7
Next: **Parameter Optimization** — deep dive into COBYLA vs SPSA, landscape features (local optima, barren plateaus), and strategies for choosing initial parameters.

---
*QOS Lab 6 | Prof. Chansu Yu | Cleveland State University*